# Rubric-Conditioned AES on ASAP 2.0 — Colab runner

This notebook clones the repository, prepares the official corpus, runs tests, and trains the baselines, the proposed cross-attention model, the ordinal variants and the ablations.

**Before running:** *Runtime → Change runtime type → GPU*.

In [ ]:
import torch, sys
print("python", sys.version.split()[0])
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("gpu", p.name, round(p.total_memory / 1e9, 1), "GB", f"sm_{p.major}{p.minor}")
else:
    print("WARNING: no GPU detected; training will be extremely slow.")

## 1. Clone and install

In [ ]:
import os
if not os.path.isdir("AES"):
    !git clone https://github.com/Suhas-Koheda/AES.git
%cd AES
!bash scripts/colab_setup.sh

## 2. Verify the pipeline (offline unit tests + synthetic smoke run)

In [ ]:
!pytest -q

In [ ]:
!python scripts/smoke_test.py

## 3. Inspect the data

Reports schema, prompt/score distributions, demographic coverage, token-length quantiles and truncation counts.

In [ ]:
!python scripts/inspect_dataset.py --tokenizer microsoft/deberta-v3-small

## 4. Model A — TF-IDF baseline

In [ ]:
!python scripts/run_baselines.py --only tfidf

## 5. Model B — Transformer concatenation baseline

In [ ]:
!python scripts/run_baselines.py --only transformer

## 6. Model C — Rubric cross-attention (proposed)

In [ ]:
!python scripts/run_from_config.py --config configs/cross_attention.yaml

## 7. Model D — Ordinal variants (CORAL and CE + expected score)

In [ ]:
!python scripts/run_from_config.py --config configs/ordinal_coral.yaml
!python scripts/run_from_config.py --config configs/ordinal_expected.yaml

## 8. Ablations (6 controlled rows)

In [ ]:
!python scripts/run_ablation.py

## 9. Held-out-prompt generalization

Seven runs, one per held-out prompt. Kept separate from the official benchmark table.

In [ ]:
!python scripts/run_cross_prompt.py --config configs/cross_attention.yaml

## 10. Tables and figures

In [ ]:
!python scripts/make_report.py --ablation

In [ ]:
import pandas as pd
from IPython.display import display, Image

for name in ["model_comparison", "ablation", "per_prompt_test"]:
    path = f"results/tables/{name}.csv"
    try:
        print(f"\n=== {name} ===")
        display(pd.read_csv(path))
    except FileNotFoundError:
        print("missing:", path)

for fig in ["qwk_comparison.png"]:
    try:
        display(Image(filename=f"results/figures/{fig}"))
    except FileNotFoundError:
        pass

## 11. Save artefacts to Drive (optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r results /content/drive/MyDrive/aes_results_$(date +%Y%m%d_%H%M)
!zip -rq results.zip results && ls -lh results.zip